# Apple Silicon CPU/GPU selection

Sweep exact statevector widths around MettleQ's measured GPU crossover and verify parity at each width.

The SDK reference and MettleQ calls below use the same circuit and result contract. Timing includes the complete call shown.

In [1]:
import numpy as np
from qiskit import QuantumCircuit, transpile
from qiskit.quantum_info import SparsePauliOp, Statevector
from qiskit.primitives import StatevectorEstimator, StatevectorSampler

from mettleq.integrations.qiskit import (
    MettleQBackend,
    MettleQEstimatorV2,
    MettleQSamplerV2,
)
from tutorials._support import (
    benchmark,
    emit_result,
    max_abs_error,
    phase_aligned_statevector_error,
    qiskit_selection,
    total_variation_distance,
)

In [2]:
import statistics

widths = [12, 14, 16]
rows = []
for width in widths:
    circuit = QuantumCircuit(width)
    for wire in range(width):
        circuit.ry(0.03 * (wire + 1), wire)
    for wire in range(width - 1):
        circuit.cx(wire, wire + 1)
    reference, reference_ms, _ = benchmark(lambda c=circuit: np.asarray(Statevector.from_instruction(c).data), repeats=2)
    backend = MettleQBackend(method="statevector", device="auto")
    compiled = transpile(circuit, backend, optimization_level=1)
    def run_mettleq(c=compiled, b=backend):
        return np.asarray(b.run(c, shots=1, return_statevector=True).result().data(0)["statevector"])
    candidate, mettleq_ms, _ = benchmark(run_mettleq, repeats=2)
    method, device = qiskit_selection(backend)
    rows.append({
        "width": width,
        "reference_ms": reference_ms,
        "mettleq_ms": mettleq_ms,
        "error": phase_aligned_statevector_error(reference, candidate),
        "method": method,
        "device": device,
    })

passed = all(row["error"] <= 3e-6 for row in rows) and rows[-1]["device"] == "gpu"
tutorial_result = emit_result(
    notebook="qiskit/15_apple_gpu_scaling.ipynb",
    framework="qiskit",
    reference_ms=statistics.median(row["reference_ms"] for row in rows),
    mettleq_ms=statistics.median(row["mettleq_ms"] for row in rows),
    check="per-width statevector atol=3e-6 and policy-selected GPU",
    passed=passed,
    exact_match=all(row["error"] == 0.0 for row in rows),
    selected_method=rows[-1]["method"],
    selected_device=rows[-1]["device"],
    metrics={"widths": rows},
    notes="The aggregate medians summarize different widths; use the per-width rows for timing interpretation.",
)

TUTORIAL_RESULT::{"check": "per-width statevector atol=3e-6 and policy-selected GPU", "exact_match": false, "framework": "qiskit", "machine": "arm64", "metrics": {"widths": [{"device": "cpu", "error": 1.0412182105401513e-07, "method": "statevector", "mettleq_ms": 0.9884165046969429, "reference_ms": 0.6587079988094047, "width": 12}, {"device": "gpu", "error": 1.3858108138808944e-07, "method": "statevector", "mettleq_ms": 2.1048954949947074, "reference_ms": 1.4170835056575015, "width": 14}, {"device": "gpu", "error": 9.131542633156187e-08, "method": "statevector", "mettleq_ms": 3.471666990662925, "reference_ms": 4.250521000358276, "width": 16}]}, "mettleq_median_ms": 2.1048954949947074, "notebook": "qiskit/15_apple_gpu_scaling.ipynb", "notes": "The aggregate medians summarize different widths; use the per-width rows for timing interpretation.", "passed": true, "python": "3.13.2", "reference_median_ms": 1.4170835056575015, "reference_over_mettleq": 0.6732322383829629, "schema_version": 1,